# atari_10m — DQN vs. Reference DQN on two Atari games (10M frames)

One plot per game: the `dqn_atari` component sweeps `ENV_HYPERS.GAME` over
`battle_zone`, `ms_pacman` at 3 seeds each. Each plot shows all 3 seeds as
faint `tab:blue` curves with the across-seed mean as a thicker `tab:blue`
curve, overlaid against a dashed black **Reference DQN** curve (see below),
clipped to our 10M-frame training window. (Atari is cluster-scale - this
notebook expects results produced elsewhere; see the experiment README.)

The `dqn_real_atari` component runs the same games under the "real Atari"
protocol (`LIMITED_ACTION_SPACE=False`, `EPISODIC_LIFE=False`,
`NOOP_MAX=0`, unclipped reward) - its seeds and mean are overlaid on the
same per-game plot in `tab:red`, alongside `dqn_atari`'s `tab:blue`. No
Reference DQN overlay for it, since that baseline is not comparable under
this protocol.

**Reference DQN** is the mean of Dopamine's published "DQN (Adam + MSE in
JAX)" baseline over its 5 seeds, from
[`baselines/atari/data`](https://github.com/google/dopamine/tree/master/baselines/atari/data)
in [google/dopamine](https://github.com/google/dopamine) (Castro et al.,
["Dopamine: A Research Framework for Deep Reinforcement
Learning"](https://arxiv.org/abs/1812.06110), 2018). Cached locally as
`dopamine_dqn_reference.json`, whose `_source` key records the exact
per-game URLs it was pulled from.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/atari_10m")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import seed_grids_for, mean_over_seeds, style

from config import EXPERIMENT

FONT_SIZE = 20   # 2x matplotlib's default (10), passed to every text call below


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


def env_title(game):
    """The plot title for one game: e.g. "Battle Zone"."""
    return game.replace("_", " ").title()


def plot_seeds_and_mean(ax, grid, stack, label, color):
    """Draw every seed as a faint curve plus the across-seed mean, thicker."""
    for row in stack:
        ax.plot(grid, row, lw=1, color=color, alpha=0.15)
    mean = mean_over_seeds(stack)
    m = ~np.isnan(mean)
    ax.plot(grid[m], mean[m], lw=1.75, color=color, label=label)


_DOPAMINE_REFERENCE = json.loads(
    (_EXP_DIR / "dopamine_dqn_reference.json").read_text()
)


def dopamine_reference_curve(game):
    """Dopamine's published DQN (Adam + MSE in JAX) curve for `game`.

    Mean raw return over its 5 published seeds, one point per training
    iteration (1M frames each); see dopamine_dqn_reference.json.
    """
    ref = _DOPAMINE_REFERENCE[game]
    return np.array(ref["frames"]), np.array(ref["mean_return"])

In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else _EXP_DIR / "results"

COMPONENT = "dqn_atari"
REAL_COMPONENT = "dqn_real_atari"

GRID = grid_for(COMPONENT)
GRID_REAL = grid_for(REAL_COMPONENT)

# GRID is in env steps; scale to frames (steps x frameskip) for the x-axis.
FRAMESKIP = load_runs(EXPERIMENT, COMPONENT)["ENV_HYPERS.FRAMESKIP"][0]
FRAME_GRID = GRID * FRAMESKIP
FRAME_GRID_REAL = GRID_REAL * FRAMESKIP  # same frameskip as dqn_atari
FRAME_TICKS = np.array([2, 4, 6, 8, 10]) * 1_000_000
FRAME_TICK_LABELS = [f"{t // 1_000_000}M" for t in FRAME_TICKS]


In [ ]:
figures = {}
df = load_runs(EXPERIMENT, COMPONENT)
df_real = load_runs(EXPERIMENT, REAL_COMPONENT)
for game in df["ENV_HYPERS.GAME"].unique(maintain_order=True):
    run_ids = df.filter(df["ENV_HYPERS.GAME"] == game)["run_id"].to_list()
    stack = seed_grids_for(EXPERIMENT, COMPONENT, GRID, run_ids=run_ids)
    real_run_ids = df_real.filter(
        df_real["ENV_HYPERS.GAME"] == game
    )["run_id"].to_list()
    stack_real = seed_grids_for(
        EXPERIMENT, REAL_COMPONENT, GRID_REAL, run_ids=real_run_ids
    )

    fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
    plot_seeds_and_mean(ax, FRAME_GRID, stack, "DQN", "tab:blue")
    plot_seeds_and_mean(
        ax, FRAME_GRID_REAL, stack_real, "DQN (real Atari)", "tab:red"
    )
    ref_frames, ref_return = dopamine_reference_curve(game)
    in_range = ref_frames <= FRAME_GRID[-1]  # match our training window
    ax.plot(
        ref_frames[in_range], ref_return[in_range], ls="--", color="black",
        lw=2.5, label="Reference DQN",
    )
    ax.set_title(env_title(game), fontsize=FONT_SIZE)
    ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
    style(ax, xlabel="Frames")   # score range differs per game, so let it auto-scale
    ax.set_ylabel(
        "Return", rotation=90, ha="center", va="center",
        labelpad=20, fontsize=FONT_SIZE,
    )
    ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
    ax.set_xticks(FRAME_TICKS)
    ax.set_xticklabels(FRAME_TICK_LABELS)
    ax.tick_params(labelsize=FONT_SIZE)
    fig.tight_layout()
    figures[f"{COMPONENT}_{game}"] = fig

plt.show()

In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")
